# EDA — WC 2026 Fixtures & Live Results
**Sources**: 
- `openfootball_2026/worldcup_2026.json` — 104 fixtures with results where played
- `wc2026_results/` — unofficial Kaggle dataset: `teams.csv`, `matches.csv`, `tournament_stages.csv`, `host_cities.csv`, SQLite DB

**Purpose**: Understand the fixture structure, which matches have been played, group standings, and what prediction targets remain. This is the submission target dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sqlite3
from pathlib import Path
import datetime

sns.set_theme(style='whitegrid', palette='muted')
OF_DATA = Path('../data/raw/openfootball_2026')
WC_DATA = Path('../data/raw/wc2026_results')
TODAY = datetime.date(2026, 6, 12)

## 1. OpenFootball — Load & Parse

In [ ]:
with open(OF_DATA / 'worldcup_2026.json') as f:
    wc_json = json.load(f)

print('Top-level keys:', list(wc_json.keys()))
print('Tournament name:', wc_json.get('name'))
print('Total matches in JSON:', len(wc_json['matches']))
print('\nSample match:')
print(json.dumps(wc_json['matches'][0], indent=2))

In [ ]:
rows = []
for m in wc_json['matches']:
    score = m.get('score', {})
    ft = score.get('ft', [None, None])
    ht = score.get('ht', [None, None])
    rows.append({
        'round': m.get('round'),
        'date': m.get('date'),
        'time': m.get('time'),
        'team1': m.get('team1'),
        'team2': m.get('team2'),
        'group': m.get('group'),
        'ground': m.get('ground'),
        'home_score_ft': ft[0] if ft else None,
        'away_score_ft': ft[1] if ft else None,
        'home_score_ht': ht[0] if ht else None,
        'away_score_ht': ht[1] if ht else None,
        'goals1': len(m.get('goals1', [])),
        'goals2': len(m.get('goals2', [])),
    })

fixtures = pd.DataFrame(rows)
fixtures['date'] = pd.to_datetime(fixtures['date'])
fixtures['played'] = fixtures['home_score_ft'].notna()
print('Fixtures shape:', fixtures.shape)
print('Matches played:', fixtures['played'].sum())
print('Matches remaining:', (~fixtures['played']).sum())
fixtures.head(5)

## 2. Tournament Structure

In [ ]:
print('Rounds:')
print(fixtures['round'].value_counts())
print()
print('Groups:')
group_stage = fixtures[fixtures['group'].notna()]
print(group_stage['group'].value_counts().sort_index())
print()
print('Venues (top 10):')
print(fixtures['ground'].value_counts().head(10))

## 3. Played Matches — Score Distribution

In [ ]:
played = fixtures[fixtures['played']].copy()
print(f'Played matches: {len(played)}')
print(f'Date range: {played["date"].min().date()} -> {played["date"].max().date()}')

if len(played) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, col, label, color in [
        (axes[0], 'home_score_ft', 'Home Goals (WC 2026)', 'steelblue'),
        (axes[1], 'away_score_ft', 'Away Goals (WC 2026)', 'coral')
    ]:
        played[col].value_counts().sort_index().plot(kind='bar', ax=ax, color=color)
        ax.set_title(label)
        ax.set_xlabel('Goals')

    plt.tight_layout()
    plt.show()

    print(f'Mean home goals: {played["home_score_ft"].mean():.2f}')
    print(f'Mean away goals: {played["away_score_ft"].mean():.2f}')
    
    results_so_far = []
    for _, r in played.iterrows():
        if r.home_score_ft > r.away_score_ft:
            results_so_far.append('team1_win')
        elif r.home_score_ft == r.away_score_ft:
            results_so_far.append('draw')
        else:
            results_so_far.append('team2_win')
    from collections import Counter
    print('Results so far:', Counter(results_so_far))

## 4. Group Stage Progress

In [ ]:
group_played = group_stage[group_stage['played']]
group_remaining = group_stage[~group_stage['played']]

print(f'Group stage matches played: {len(group_played)} / {len(group_stage)}')
print(f'Group stage matches remaining: {len(group_remaining)}')
print()
print('Group stage progress:')
progress = group_stage.groupby('group').agg(
    total=('played', 'count'),
    played=('played', 'sum')
).assign(remaining=lambda x: x.total - x.played)
print(progress.sort_index())

## 5. Upcoming Matches (Prediction Targets)

In [ ]:
remaining = fixtures[~fixtures['played']].sort_values('date')
print(f'Total remaining fixtures to predict: {len(remaining)}')
print()
print('Next 10 upcoming matches:')
print(remaining[['date', 'round', 'group', 'team1', 'team2', 'ground']].head(10).to_string(index=False))

print()
print('Remaining by stage:')
def classify_stage(row):
    if pd.notna(row['group']):
        return 'Group stage'
    return row['round'] if pd.notna(row['round']) else 'Unknown'

remaining['stage'] = remaining.apply(classify_stage, axis=1)
print(remaining['stage'].value_counts())

## 6. Unofficial Kaggle Dataset — Cross-check

In [ ]:
teams_df = pd.read_csv(WC_DATA / 'teams.csv')
matches_df = pd.read_csv(WC_DATA / 'matches.csv')
stages_df = pd.read_csv(WC_DATA / 'tournament_stages.csv')
cities_df = pd.read_csv(WC_DATA / 'host_cities.csv')

print('Teams:', teams_df.shape)
print(teams_df.head())
print()
print('Matches:', matches_df.shape)
print(matches_df.head())
print()
print('Stages:', stages_df)
print()
print('Cities:', cities_df.shape)
print(cities_df.head())

In [ ]:
# SQLite DB check
conn = sqlite3.connect(WC_DATA / 'worldcup2026.db')
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print('SQLite tables:', [t[0] for t in tables])
for t_name, in tables:
    df = pd.read_sql(f'SELECT * FROM {t_name} LIMIT 3', conn)
    print(f'\n{t_name}:', df.shape)
    print(df)
conn.close()

In [ ]:
# Teams coverage check: does Kaggle dataset have all 48 teams?
kaggle_teams = set(teams_df[teams_df['is_placeholder'] == 0]['team_name'].tolist())
openfootball_teams = set(fixtures['team1'].tolist() + fixtures['team2'].tolist())

print('Teams in Kaggle dataset (non-placeholder):', len(kaggle_teams))
print('Teams in OpenFootball:', len(openfootball_teams))
print()
in_kaggle_not_of = kaggle_teams - openfootball_teams
in_of_not_kaggle = openfootball_teams - kaggle_teams
if in_kaggle_not_of:
    print('In Kaggle but not OpenFootball:', sorted(in_kaggle_not_of)[:10])
if in_of_not_kaggle:
    print('In OpenFootball but not Kaggle:', sorted(in_of_not_kaggle)[:10])
print('Note: Name mismatches expected — need harmonization before model training')

## 7. Red Flags Summary

In [ ]:
red_flags = []

if len(played) == 0:
    red_flags.append('OpenFootball JSON shows no played matches — may need manual update')

if in_of_not_kaggle:
    red_flags.append(f'Team name mismatches between OpenFootball and Kaggle: {sorted(in_of_not_kaggle)[:5]} — need harmonization table')

placeholder_count = (teams_df['is_placeholder'] == 1).sum()
if placeholder_count > 0:
    red_flags.append(f'Kaggle dataset has {placeholder_count} placeholder teams (knockout TBDs) — expected for group stage')

if red_flags:
    print('RED FLAGS:')
    for f in red_flags:
        print(' -', f)
else:
    print('No red flags — fixture data looks complete.')